##### Autonomous App Architecture: A Multi-Agent Framework for Concept Development.

Using AutoGen to Create a Multi-Agent System for the development and review of a mobile app concept. This will involve brainstorming an app idea, creating a basic product specification, and reviewing it from various perspectives. 

In [1]:
import getpass
import os
import autogen
from autogen import ConversableAgent

In [2]:
# This function prompts the user for input, but with a twist: it hides what you type.
os.environ["OPEN_API_KEY"] = getpass.getpass()

In [3]:
llm_config = {"model": "gpt-3.5-turbo"}

In [4]:
task='''
Create a concept for an innovative mobile app that solves a real world problem. 
Provide a brief description of the apps main features and target audience.
'''

In [7]:
app_developer = autogen.AssistantAgent(
    name="App_Developer",
    system_message="You are a creative app developer."
    "You brainstorm innovative mobile app ideas and create concise product specifications."
    "You must refine your concept based on the feedback you receive and provide an updated version."
    "Only return your final work without additional comments.",
    llm_config=llm_config
)

In [9]:
product_manager = autogen.AssistantAgent(
    name="Product_Manager",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a product manager. You review app concepts"
    "and provide constructive feedback to improve the product specification. "
)

In [10]:
ux_designer = autogen.AssistantAgent(
    name="UX_Designer",
    llm_config=llm_config,
    system_message="You are a UX designer known for your ability to improve "
    "user experience and interface design. Ensure your suggestions are concise"
    "(within 3 bullet points), concrete and to the point. Begin the review by stating "
    "your role."
)

In [11]:
market_analyst = autogen.AssistantAgent(
    name="Market_Analyst",
    llm_config=llm_config,
    system_message="You are a market analyst known for your ability to "
    "assess market potential and identify target audiences. Ensure your suggestions are concise"
    "(within 3 bullet points), concrete and to the point. Begin the review by stating"
    "your role."
)

In [12]:
tech_feasibility_expert = autogen.AssistantAgent(
    name="Tech_Feasibility_Expert",
    llm_config=llm_config,
    system_message="You are a technical feasibility expert known for your ability"
    "to assess the technical challenges and the feasibility of app concepts."
    "Ensure your suggestions are concise (within 3 bullet points), concrete and to the point. Begin the review by stating"
    "your role."
)

In [14]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer. You aggregate and review"
    "the work of other reviewers and give a final suggestion on the app concept."
)

In [15]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following app concept.
    \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']} '''

In [16]:
review_chats = [
    {
        "recipient": ux_designer,
        "message": reflection_message,        
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return review as json object only:"
            "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",
        },
        "max_turns": 1,        
    },
    {
        "recipient": market_analyst,
        "message": reflection_message,        
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return review as json object only:"
            "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",
        },
        "max_turns": 1,        
    },
    {
        "recipient": tech_feasibility_expert,
        "message": reflection_message,        
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return review as json object only:"
            "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",
        },
        "max_turns": 1,        
    },
    {
        "recipient": meta_reviewer,
        "message": "Aggregate feedback from all reviewers and give final suggestions on the app concept",                
        "max_turns": 1,        
    },
]

In [17]:
product_manager.register_nested_chats(
    review_chats,
    trigger=app_developer
)

In [20]:
res = product_manager.initiate_chat(
    recipient=app_developer,\
    message=task,
    max_turns=2,
    summary_method="last_msg"
)
print(res.summary)

Product_Manager (to App_Developer):


Create a concept for an innovative mobile app that solves a real world problem. 
Provide a brief description of the apps main features and target audience.


--------------------------------------------------------------------------------
App_Developer (to Product_Manager):

App Name: FitTime

Description:
FitTime is a mobile app designed to help busy professionals prioritize their health and fitness amidst their demanding schedules. The app offers personalized workout plans and nutrition guidance tailored to individual needs and goals. Users can access quick and effective at-home workouts, healthy recipes, progress tracking tools, and mindfulness practices to achieve a balanced lifestyle.

Main Features:
1. Personalized workout plans: Users input their fitness goals, schedule, and preferences to receive customized workout routines.
2. Nutrition guidance: Access to healthy recipes, meal plans, and nutritional tips to support overall health goals.
3